In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, desc, broadcast
import time

'''
Change dynamicAllocation.maxExecutors to simulate horizontal scaling for that number of executors
Change executor.instances to mimic the dynamicAllocation.maxExecutors.
executor.cores = 4 to mimic the max cores in Chenyang's Spark-cluster.
executor.memory = 4g, fixed to 4GiB to limit combinations of scaling, do not alter.
driver.cores = 4, fixed to 4GiB, limit combinations of scaling, do not alter. 
driver.memory = 4g, fixed to 4GiB, limit combinations of scaling, do not alter. 
cores.max = 4, adapt to match the number of executors used.
'''

spark = SparkSession.builder\
    .master("spark://192.168.2.156:7077") \
    .appName("Jakob_Ekholm_Project")\
    .config("spark.dynamicAllocation.enabled", True)\
    .config("spark.dynamicAllocation.maxExecutors", 1)\
    .config("spark.executor.instances", 1)\
    .config("spark.executor.cores", 4)\
    .config("spark.executor.memory", "4g")\
    .config("spark.driver.cores", 4)\
    .config("spark.driver.memory", "4g")\
    .config("spark.cores.max", 4)\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [11]:

start_time = time.time()

# Read data
reddit = spark.read.json("hdfs://192.168.2.156:9000/data/reddit/reddit_100k.json") # Choose which dataset to load
name_df = spark.read.csv("hdfs://192.168.2.156:9000/output/group31/Formatted_Celebrities.csv", 
                         header=True, inferSchema=True).select(col("name").alias("name"))

reddit_selected = reddit.select("normalizedBody", "subreddit", "summary").dropna()
reddit_selected = reddit_selected.withColumn("normalizedBody", lower(col("normalizedBody")))
reddit_selected = reddit_selected.repartition(16)  # Distribute workload

# Use Broadcast Join Instead of Cross Join
joined_df = reddit_selected.join(broadcast(name_df), reddit_selected["normalizedBody"].contains(name_df["name"]))

# Count occurrences
count_df = joined_df.groupBy("name").count().withColumnRenamed("count", "times")
sorted_df = count_df.orderBy(desc("times"))

# Save results
output_path = "hdfs://192.168.2.156:9000/output/group31/celebrity_counts_sorted.csv"
sorted_df.write.csv(output_path, header=True, mode="overwrite")

# Print the top 5 most frequently mentioned names
print("Top 5 Most Mentioned Celebrities:")
sorted_df.show(5)

end_time = time.time()
print(f"Execution Time: {end_time - start_time:.2f} seconds")


Top 5 Most Mentioned Celebrities:


[Stage 78:====================================================>   (15 + 1) / 16]

+--------+-----+
|    name|times|
+--------+-----+
|     nas| 1737|
|    pink|  471|
|   usher|  134|
|   drake|   78|
|ron paul|   67|
+--------+-----+
only showing top 5 rows

Execution Time: 68.34 seconds
